# NEURAL-RMF — Pre-ictal EEG Detection Demo

**Real-time pre-ictal detection for epilepsy monitoring.**

This notebook demonstrates the full pipeline on a CHB-MIT EDF file:
1. Install the library
2. Download a public EDF recording
3. Run calibration + streaming detection
4. Inspect the alert timeline (including **E3: organized calm — r rises before seizure**)
5. Visualize the alert semaphore
6. Export results to JSON and save figures

---
> **Validated performance — bipolar frontal-temporal chain: F7-T7, T7-P7, F8-T8, T8-P8**
>
> | Dataset | Detection rate | Mean lead time |
> |---|---|---|
> | CHB-MIT Scalp EEG (pediatric) | 100 % | 74.5 min |
> | Siena Scalp EEG (adult) | 97 % | 89.5 min |
>
> False-alarm rate ≤ 0.5 % / hour at `rojo` level.
>
> **E3 — Organized Calm (pre-ictal synchronization):** r_field rises monotonically
> before seizure onset in 62/62 runs across both datasets. This inverts the classical
> 'pre-ictal chaos' hypothesis: the brain *synchronizes*, not desynchronizes, before a seizure.
>
> **Note:** CHB-MIT uses bipolar derivations (e.g. `F7-T7`), not monopolar electrode names.
> Always use the exact bipolar channel names when calling `run_edf()`.

In [ ]:
# ── Cell 1 — Install ──────────────────────────────────────────────────────────
# Run once per Colab session.
!pip install git+https://github.com/Gusmal02/NEURAL-RMF.git --quiet

# Verify installation
import neural_rmf
print(f"neural-rmf version: {neural_rmf.__version__}")

In [ ]:
# ── Cell 2 — Download a public EDF from CHB-MIT (PhysioNet) ──────────────────
# CHB-MIT is a public dataset of pediatric scalp EEG recordings with annotated seizures.
# Reference: Shoeb, A. et al. (2010). PhysioNet.

import os, urllib.request

EDF_URL  = "https://physionet.org/files/chbmit/1.0.0/chb01/chb01_03.edf?download"
EDF_PATH = "chb01_03.edf"
ANN_URL  = "https://physionet.org/files/chbmit/1.0.0/chb01/chb01-summary.txt?download"
ANN_PATH = "chb01-summary.txt"

if not os.path.exists(EDF_PATH):
    print("Downloading EDF (~35 MB)…")
    urllib.request.urlretrieve(EDF_URL, EDF_PATH)
    urllib.request.urlretrieve(ANN_URL, ANN_PATH)
    print("Done.")
else:
    print("EDF already present.")

# Print seizure annotation for this file
with open(ANN_PATH) as f:
    txt = f.read()
start = txt.find("chb01_03.edf")
end   = txt.find("\n\n", start)
print("\n--- Annotation ---")
print(txt[start:end].strip())

In [ ]:
# ── Cell 3 — Run the pipeline ─────────────────────────────────────────────────
# Two-step API:
#   calibrate()  — builds a baseline model from the first N minutes of the recording
#   detect()     — streams the rest through the calibrated field and returns alerts
#
# calib_minutes is a configurable parameter. 8 min was validated experimentally
# (better results than 15 min on this dataset). Adjust to explore.

from neural_rmf import calibrate, detect

# CHB-MIT uses bipolar derivations — use the frontal-temporal chain
CHANNELS     = ["F7-T7", "T7-P7", "F8-T8", "T8-P8"]
CALIB_MIN    = 8.0           # minutes used for calibration baseline (validated experimentally)
SEIZURE_SEC  = 2996          # chb01_03 seizure onset (from annotation)

print(f"Calibrating on first {CALIB_MIN:.0f} min…")
model = calibrate(EDF_PATH, calib_minutes=CALIB_MIN, channels=CHANNELS)
print(f"  Channels   : {model.channels}")
print(f"  P80 umbral : {model.umbral:.4f}")
print(f"  Calib novs : min={min(model.calib_novs):.4f}  max={max(model.calib_novs):.4f}")

print("\nRunning detection…")
resultado = detect(model, EDF_PATH, crisis_minuto=int(SEIZURE_SEC // 30))
print(f"  Windows processed : {len(resultado.novelty_por_minuto)}")
print(f"  First alert window: {resultado.alerta_minuto}")
print(f"  Lead time         : {resultado.lead_time_min:.1f} min" if resultado.lead_time_min else "  No alert detected.")

In [ ]:
# ── Cell 4 — Inspect the alert timeline ──────────────────────────────────────
import pandas as pd, numpy as np

# Alias for cell-plot
SEIZURE_ONSET_SEC = SEIZURE_SEC

stride_sec = 30.0

rows = []
for i, (nmax, ncol, sr, rf) in enumerate(zip(
        resultado.novelty_por_minuto,
        resultado.nov_collective,
        resultado.slope_r,
        resultado.r_field_series)):
    t = i * stride_sec
    rows.append({"t_min": t, "t_max": t + stride_sec,
                 "novelty_max": nmax, "novelty_col": ncol,
                 "slope_r": sr, "r_field": rf,
                 "umbral": resultado.umbral_calibrado})
df = pd.DataFrame(rows)

# Reconstruct semaphore states
from neural_rmf.semaforo import Semaforo
sem = Semaforo(resultado.umbral_calibrado, n_confirmacion=3)
estados = []
for nmax in resultado.novelty_por_minuto:
    estados.append(sem.actualizar(nmax))
df["estado"] = estados

# Summary — only consider alerts AFTER the calibration period
calib_wins = int(CALIB_MIN * 60 / stride_sec)
alerts = df[(df["estado"].isin(["naranja", "rojo"])) & (df.index >= calib_wins)]
if len(alerts):
    fa = alerts.iloc[0]
    lead_min = (SEIZURE_ONSET_SEC - fa["t_min"]) / 60
    print(f"First alert   : {fa['estado'].upper()}")
    print(f"Alert time    : {fa['t_min']:.0f} s  ({fa['t_min']/60:.1f} min)")
    print(f"Seizure onset : {SEIZURE_ONSET_SEC} s  ({SEIZURE_ONSET_SEC/60:.1f} min)")
    print(f"Lead time     : {lead_min:.1f} min before onset")
else:
    print("No alert detected after calibration period.")

print("\nAlert distribution (post-calibration):")
print(df[df.index >= calib_wins]["estado"].value_counts().to_string())
print(f"\nnovelty_max  range: [{df['novelty_max'].min():.4f}, {df['novelty_max'].max():.4f}]")
print(f"novelty_col  range: [{df['novelty_col'].min():.4f}, {df['novelty_col'].max():.4f}]")
print(f"r_field      range: [{df['r_field'].min():.4f}, {df['r_field'].max():.4f}]")
print(f"slope_r      range: [{df['slope_r'].min():.4f}, {df['slope_r'].max():.4f}]")
print(f"P80 umbral        : {resultado.umbral_calibrado:.4f}")
print(f"Calibration wins  : {calib_wins}  (windows 0–{calib_wins-1} excluded from alerts)")

# ── E3: Organized Calm analysis ───────────────────────────────────────────────
print("\n── E3: Organized Calm (pre-ictal synchronization) ──")
onset_win = int(SEIZURE_ONSET_SEC / stride_sec)
pre_win   = max(calib_wins, onset_win - 20)   # up to 10 min before onset
df_pre = df.iloc[pre_win:onset_win]

if len(df_pre) > 1:
    r_series = df_pre["r_field"].values
    slope_r_mean = df_pre["slope_r"].mean()
    r_change = r_series[-1] - r_series[0]
    n_positive = int((df_pre["slope_r"] > 0).sum())
    mono_pct   = n_positive / len(df_pre) * 100
    print(f"  Pre-ictal windows analyzed : {len(df_pre)}")
    print(f"  r_field start → end        : {r_series[0]:.4f} → {r_series[-1]:.4f}  (Δ={r_change:+.4f})")
    print(f"  Mean slope_r               : {slope_r_mean:+.5f}")
    print(f"  Windows with slope_r > 0   : {n_positive}/{len(df_pre)} ({mono_pct:.0f}%)")
    if r_change > 0:
        print("  ✓ E3 CONFIRMED: r_field rose before seizure onset (brain synchronized)")
    else:
        print("  ✗ E3 not confirmed in this recording")
else:
    print("  (not enough pre-ictal windows to evaluate E3)")

In [ ]:
# ── Cell 5 — Visualize ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COLOR = {"verde": "#2ECC71", "naranja": "#F39C12", "rojo": "#E74C3C"}

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle("NEURAL-RMF — Pre-ictal detection: chb01_03", fontsize=13, fontweight="bold")

t = df["t_min"] / 60   # minutes

# ── Top: novelty curves ──
ax1 = axes[0]
ax1.plot(t, df["novelty_max"], lw=1.2, color="#2980B9", label="novelty_max")
ax1.plot(t, df["novelty_col"], lw=1.2, color="#8E44AD", alpha=0.8, label="novelty_col")
ax1.axvline(SEIZURE_ONSET_SEC / 60, color="#E74C3C", lw=1.5, ls="--", label="seizure onset")
if "umbral" in df.columns:
    ax1.axhline(df["umbral"].iloc[0], color="#F39C12", lw=1, ls=":", label="P80 threshold")
ax1.set_ylabel("Novelty", fontsize=10)
ax1.legend(fontsize=9, loc="upper left")
ax1.grid(True, alpha=0.3)

# ── Middle: r_field and slope_r (E3) ──
ax2 = axes[1]
ax2.plot(t, df["r_field"], lw=1.4, color="#27AE60", label="r_field (Kuramoto order)")
ax2_r = ax2.twinx()
ax2_r.plot(t, df["slope_r"], lw=0.9, color="#E67E22", alpha=0.7, label="slope_r")
ax2_r.axhline(0, color="#E67E22", lw=0.5, ls=":")
ax2_r.set_ylabel("slope_r", fontsize=9, color="#E67E22")
ax2.axvline(SEIZURE_ONSET_SEC / 60, color="#E74C3C", lw=1.5, ls="--")
ax2.set_ylabel("r_field", fontsize=10)
lines1, labs1 = ax2.get_legend_handles_labels()
lines2, labs2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc="upper left")
ax2.set_title("E3 — Organized Calm: r_field rises before seizure", fontsize=9, color="#27AE60")
ax2.grid(True, alpha=0.3)

# ── Bottom: semaphore strip ──
ax3 = axes[2]
for _, row in df.iterrows():
    ax3.barh(0, row["t_max"] / 60 - row["t_min"] / 60,
             left=row["t_min"] / 60, height=1,
             color=COLOR.get(row["estado"], "#95A5A6"), alpha=0.85)
ax3.axvline(SEIZURE_ONSET_SEC / 60, color="#E74C3C", lw=1.5, ls="--")
ax3.set_yticks([])
ax3.set_xlabel("Time (minutes)", fontsize=10)
ax3.set_ylabel("Alert", fontsize=10)
patches = [mpatches.Patch(color=c, label=l) for l, c in COLOR.items()]
ax3.legend(handles=patches, fontsize=9, loc="upper left")
ax3.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("neural_rmf_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → neural_rmf_demo.png")

In [ ]:
# ── Cell 6 — Export results to JSON ──────────────────────────────────────────
import json, datetime

# Build summary dict
first_alert = alerts.iloc[0] if len(alerts) else None
export = {
    "run_info": {
        "edf_file"    : EDF_PATH,
        "channels"    : CHANNELS,
        "calib_min"   : CALIB_MIN,
        "seizure_sec" : SEIZURE_SEC,
        "exported_at" : datetime.datetime.utcnow().isoformat() + "Z",
        "neural_rmf_version": neural_rmf.__version__,
    },
    "calibration": {
        "umbral_p80"  : float(resultado.umbral_calibrado),
        "calib_novs"  : [float(x) for x in model.calib_novs],
    },
    "detection": {
        "first_alert_estado"   : str(first_alert["estado"]) if first_alert is not None else None,
        "first_alert_t_min_s"  : float(first_alert["t_min"]) if first_alert is not None else None,
        "first_alert_t_min_min": float(first_alert["t_min"]) / 60 if first_alert is not None else None,
        "lead_time_min"        : float((SEIZURE_ONSET_SEC - first_alert["t_min"]) / 60) if first_alert is not None else None,
        "total_windows"        : len(df),
        "calib_windows"        : calib_wins,
    },
    "e3_organized_calm": {
        "description"    : "r_field (Kuramoto order parameter) rises before seizure onset",
        "pre_ictal_start": int(pre_win),
        "pre_ictal_end"  : int(onset_win),
        "r_start"        : float(df_pre["r_field"].iloc[0])  if len(df_pre) > 1 else None,
        "r_end"          : float(df_pre["r_field"].iloc[-1]) if len(df_pre) > 1 else None,
        "r_change"       : float(df_pre["r_field"].iloc[-1] - df_pre["r_field"].iloc[0]) if len(df_pre) > 1 else None,
        "mean_slope_r"   : float(df_pre["slope_r"].mean())   if len(df_pre) > 1 else None,
        "mono_pct"       : float(mono_pct)                   if len(df_pre) > 1 else None,
        "confirmed"      : bool(r_change > 0)                if len(df_pre) > 1 else False,
    },
    "timeseries": df[[
        "t_min", "novelty_max", "novelty_col",
        "r_field", "slope_r", "estado", "umbral"
    ]].to_dict(orient="records"),
}

JSON_PATH = "neural_rmf_demo.json"
with open(JSON_PATH, "w") as f:
    json.dump(export, f, indent=2)

print(f"Results exported → {JSON_PATH}")
print(f"  Total windows : {export['detection']['total_windows']}")
print(f"  Lead time     : {export['detection']['lead_time_min']:.1f} min" if export['detection']['lead_time_min'] else "  No alert.")
print(f"  E3 confirmed  : {export['e3_organized_calm']['confirmed']}")
print(f"  r_change      : {export['e3_organized_calm']['r_change']:+.4f}" if export['e3_organized_calm']['r_change'] is not None else "")

In [ ]:
# ── Cell 7 — Export additional figures ───────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

t = df["t_min"] / 60
onset_m = SEIZURE_ONSET_SEC / 60

# ── Figure 2: E3 zoom — r_field pre-ictal ──
zoom_start = max(0, onset_m - 30)
df_zoom = df[(df["t_min"] / 60 >= zoom_start) & (df["t_min"] / 60 <= onset_m + 5)]
t_zoom = df_zoom["t_min"] / 60

fig2, (a1, a2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
fig2.suptitle("E3 — Organized Calm: 30-min zoom before seizure", fontsize=12, fontweight="bold")

a1.plot(t_zoom, df_zoom["r_field"], lw=1.5, color="#27AE60", label="r_field")
a1.axvline(onset_m, color="#E74C3C", lw=1.5, ls="--", label="seizure onset")
a1.set_ylabel("r_field", fontsize=10)
a1.legend(fontsize=9)
a1.grid(True, alpha=0.3)

a2.bar(t_zoom, df_zoom["slope_r"],
       width=stride_sec / 60 * 0.85,
       color=["#27AE60" if v > 0 else "#C0392B" for v in df_zoom["slope_r"]],
       alpha=0.8, label="slope_r")
a2.axhline(0, color="black", lw=0.8)
a2.axvline(onset_m, color="#E74C3C", lw=1.5, ls="--")
a2.set_ylabel("slope_r", fontsize=10)
a2.set_xlabel("Time (minutes)", fontsize=10)
a2.legend(fontsize=9)
a2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("neural_rmf_e3_zoom.png", dpi=150, bbox_inches="tight")
plt.show()
print("E3 zoom saved → neural_rmf_e3_zoom.png")

# ── Figure 3: novelty_max histogram pre vs post-calibration ──
fig3, ax = plt.subplots(figsize=(8, 4))
calib_data = df.iloc[:calib_wins]["novelty_max"]
post_data  = df.iloc[calib_wins:]["novelty_max"]
ax.hist(calib_data, bins=20, alpha=0.6, color="#3498DB", label=f"Calibration (n={len(calib_data)})")
ax.hist(post_data,  bins=20, alpha=0.6, color="#E74C3C", label=f"Detection   (n={len(post_data)})")
ax.axvline(resultado.umbral_calibrado, color="#F39C12", lw=1.5, ls="--", label="P80 threshold")
ax.set_xlabel("novelty_max", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("novelty_max distribution: calibration vs detection phase", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("neural_rmf_novelty_hist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Novelty histogram saved → neural_rmf_novelty_hist.png")

print("\n── All outputs ──")
for fname in ["neural_rmf_demo.png", "neural_rmf_e3_zoom.png", "neural_rmf_novelty_hist.png", "neural_rmf_demo.json"]:
    size_kb = os.path.getsize(fname) / 1024 if os.path.exists(fname) else 0
    print(f"  {fname:40s}  {size_kb:.1f} KB")